In [1]:
import pandas as pd

# ============================================================
# 1. CHARGEMENT DES DONNÉES
# ============================================================

df_dvf = pd.read_csv("../data/processed/dvf_nettoye.csv")

df_insee = pd.read_csv("../data/processed/insee_comparateur_territoires_69.csv")

print("Dimensions DVF :", df_dvf.shape)
print("Dimensions INSEE :", df_insee.shape)


# ============================================================
# 2. VÉRIFICATION DU CODE COMMUNE
# ============================================================

print("\nType code_commune DVF :", df_dvf["code_commune"].dtype)
print("Type code_commune INSEE :", df_insee["code_commune"].dtype)

# Harmonisation pour sécuriser la jointure
df_dvf["code_commune"] = df_dvf["code_commune"].astype(str).str.zfill(5)
df_insee["code_commune"] = df_insee["code_commune"].astype(str).str.zfill(5)


# ============================================================
# 3. VÉRIFICATION DES COMMUNES
# ============================================================

communes_dvf = set(df_dvf["code_commune"].unique())
communes_insee = set(df_insee["code_commune"].unique())

print("\nNombre de communes DVF :", len(communes_dvf))
print("Nombre de communes INSEE :", len(communes_insee))

communes_absentes = communes_dvf - communes_insee

print("Communes DVF absentes de l'INSEE :", communes_absentes)


# ============================================================
# 4. JOINTURE DVF + INSEE
# ============================================================

df_dvf_insee = df_dvf.merge(
    df_insee,
    on="code_commune",
    how="left",
    indicator=True
)

print("\nDimensions avant jointure :", df_dvf.shape)
print("Dimensions après jointure :", df_dvf_insee.shape)

print("\nRésultat de la jointure :")
print(df_dvf_insee["_merge"].value_counts())


# ============================================================
# 5. CONTRÔLE DES VALEURS MANQUANTES INSEE
# ============================================================

colonnes_insee = [
    "niveau_vie_median",
    "population",
    "superficie_km2",
    "nb_etablissements",
    "densite_population",
    "part_proprietaires",
    "part_logements_vacants"
]

print("\nValeurs manquantes après jointure :")
print(df_dvf_insee[colonnes_insee].isna().sum())


# Suppression de la colonne technique de jointure
df_dvf_insee = df_dvf_insee.drop(columns="_merge")

# Aperçu du dataset enrichi
display(df_dvf_insee.head())

Dimensions DVF : (110889, 16)
Dimensions INSEE : (275, 13)

Type code_commune DVF : int64
Type code_commune INSEE : int64

Nombre de communes DVF : 276
Nombre de communes INSEE : 275
Communes DVF absentes de l'INSEE : {'69152', '69159'}

Dimensions avant jointure : (110889, 16)
Dimensions après jointure : (110889, 29)

Résultat de la jointure :
_merge
both          110304
left_only        585
right_only         0
Name: count, dtype: int64

Valeurs manquantes après jointure :
niveau_vie_median         587
population                585
superficie_km2            585
nb_etablissements         585
densite_population        585
part_proprietaires        585
part_logements_vacants    585
dtype: int64


,valeur_fonciere,code_commune,lot1_surface_carrez,nombre_lots,type_local,surface_reelle_bati,nombre_pieces_principales,surface_terrain,longitude,latitude,...,nb_proprietaires,nb_residences_secondaires,nb_logements_vacants,niveau_vie_median,population,superficie_km2,nb_etablissements,densite_population,part_proprietaires,part_logements_vacants
0,375000.0,69385,98.86,2,Appartement,108.0,5.0,NaN,4.797315,45.754569,...,10216.77721,977.61046,2123.78484,29330.0,47777.0,6.23,1347.0,7668.860353,44.071008,8.080164
1,203050.0,69385,NaN,2,Appartement,83.0,5.0,NaN,4.786061,45.757444,...,10216.77721,977.61046,2123.78484,29330.0,47777.0,6.23,1347.0,7668.860353,44.071008,8.080164
2,140000.0,69389,NaN,2,Appartement,73.0,3.0,NaN,4.795657,45.782647,...,8781.48239,1475.41003,3395.54741,24160.0,53567.0,7.25,2471.0,7388.551724,33.308790,10.871034
3,174100.0,69389,69.63,1,Appartement,69.0,3.0,NaN,4.806255,45.763372,...,8781.48239,1475.41003,3395.54741,24160.0,53567.0,7.25,2471.0,7388.551724,33.308790,10.871034
4,470250.0,69207,NaN,0,Maison,121.0,5.0,705.0,4.807607,45.879440,...,720.30245,19.43445,55.76842,30040.0,3124.0,5.43,64.0,575.322284,55.103465,4.034219


In [2]:
# Nombre de transactions pour les communes encore absentes
print(
    df_dvf[
        df_dvf["code_commune"].isin(["69152", "69159"])
    ]["code_commune"].value_counts()
)

print(
    df_dvf[
        df_dvf["code_commune"].isin(["69152", "69159"])
    ].groupby(["code_commune", "annee_mutation"]).size()
)

code_commune
69152    421
69159    164
Name: count, dtype: int64
code_commune  annee_mutation
69152         2021              148
              2022              110
              2023              110
              2024               53
69159         2021               51
              2022               37
              2023               56
              2024               20
dtype: int64


In [3]:
import pandas as pd

# Charger les fichiers
df_dvf = pd.read_csv("../data/processed/dvf_nettoye.csv")
df_insee = pd.read_csv("../data/processed/insee_comparateur_territoires_69.csv")

# Mettre les codes au même format
df_dvf["code_commune"] = df_dvf["code_commune"].astype(str).str.zfill(5)
df_insee["code_commune"] = df_insee["code_commune"].astype(str).str.zfill(5)

# Créer un code temporaire uniquement pour la jointure
df_dvf["code_jointure_insee"] = df_dvf["code_commune"].replace({
    "69152": "69149",
    "69159": "69114"
})

# Renommer le code commune INSEE pour éviter tout conflit
df_insee = df_insee.rename(columns={"code_commune": "code_jointure_insee"})

# Jointure
df_dvf_insee = df_dvf.merge(
    df_insee,
    on="code_jointure_insee",
    how="left",
    indicator=True
)

# Vérifications
print("Dimensions DVF :", df_dvf.shape)
print("Dimensions INSEE :", df_insee.shape)
print("Dimensions après jointure :", df_dvf_insee.shape)

print("\nRésultat de la jointure :")
print(df_dvf_insee["_merge"].value_counts())

print("\nValeurs manquantes :")
print(
    df_dvf_insee[
        [
            "niveau_vie_median",
            "population",
            "densite_population",
            "part_proprietaires",
            "part_logements_vacants"
        ]
    ].isna().sum()
)

Dimensions DVF : (110889, 17)
Dimensions INSEE : (275, 13)
Dimensions après jointure : (110889, 30)

Résultat de la jointure :
_merge
both          110889
left_only          0
right_only         0
Name: count, dtype: int64

Valeurs manquantes :
niveau_vie_median         2
population                0
densite_population        0
part_proprietaires        0
part_logements_vacants    0
dtype: int64


In [4]:
# Supprimer les colonnes utilisées uniquement pour la jointure
df_dvf_insee = df_dvf_insee.drop(columns=["_merge", "code_jointure_insee"])

# Vérification finale
print("Dimensions finales :", df_dvf_insee.shape)
print("Nombre de doublons :", df_dvf_insee.duplicated().sum())

display(df_dvf_insee.head())

# Enregistrer le dataset DVF enrichi avec les données INSEE
df_dvf_insee.to_csv("../data/processed/dvf_insee_enrichi.csv", index=False)

print("Fichier dvf_insee_enrichi.csv enregistré.")

Dimensions finales : (110889, 28)
Nombre de doublons : 35


,valeur_fonciere,code_commune,lot1_surface_carrez,nombre_lots,type_local,surface_reelle_bati,nombre_pieces_principales,surface_terrain,longitude,latitude,...,nb_proprietaires,nb_residences_secondaires,nb_logements_vacants,niveau_vie_median,population,superficie_km2,nb_etablissements,densite_population,part_proprietaires,part_logements_vacants
0,375000.0,69385,98.86,2,Appartement,108.0,5.0,NaN,4.797315,45.754569,...,10216.77721,977.61046,2123.78484,29330.0,47777.0,6.23,1347.0,7668.860353,44.071008,8.080164
1,203050.0,69385,NaN,2,Appartement,83.0,5.0,NaN,4.786061,45.757444,...,10216.77721,977.61046,2123.78484,29330.0,47777.0,6.23,1347.0,7668.860353,44.071008,8.080164
2,140000.0,69389,NaN,2,Appartement,73.0,3.0,NaN,4.795657,45.782647,...,8781.48239,1475.41003,3395.54741,24160.0,53567.0,7.25,2471.0,7388.551724,33.308790,10.871034
3,174100.0,69389,69.63,1,Appartement,69.0,3.0,NaN,4.806255,45.763372,...,8781.48239,1475.41003,3395.54741,24160.0,53567.0,7.25,2471.0,7388.551724,33.308790,10.871034
4,470250.0,69207,NaN,0,Maison,121.0,5.0,705.0,4.807607,45.879440,...,720.30245,19.43445,55.76842,30040.0,3124.0,5.43,64.0,575.322284,55.103465,4.034219


Fichier dvf_insee_enrichi.csv enregistré.


In [5]:
# Récupérer les doublons stricts
doublons = df_dvf[df_dvf.duplicated(keep=False)]

# Afficher les lignes identiques côte à côte
display(
    doublons.sort_values(
        by=list(df_dvf.columns),
        na_position="first"
    )
)

print("Nombre de doublons strictement identiques :", df_dvf.duplicated().sum())

,valeur_fonciere,code_commune,lot1_surface_carrez,nombre_lots,type_local,surface_reelle_bati,nombre_pieces_principales,surface_terrain,longitude,latitude,annee_mutation,mois_mutation,prix_m2,surface_par_piece,log_surface,distance_lyon_km,code_jointure_insee
35997,48500.0,69019,NaN,1,Appartement,25.0,1.0,NaN,4.749520,46.107800,2022,11,1940.000000,25.000000,3.258097,38.793451,69019
36000,48500.0,69019,NaN,1,Appartement,25.0,1.0,NaN,4.749520,46.107800,2022,11,1940.000000,25.000000,3.258097,38.793451,69019
21617,80000.0,69072,NaN,1,Appartement,26.0,1.0,NaN,4.761246,45.821361,2021,4,3076.923077,26.000000,3.295837,8.611575,69072
21666,80000.0,69072,NaN,1,Appartement,26.0,1.0,NaN,4.761246,45.821361,2021,4,3076.923077,26.000000,3.295837,8.611575,69072
88052,82000.0,69386,17.0,1,Appartement,17.0,1.0,NaN,4.851853,45.767800,2024,10,4823.529412,17.000000,2.890372,1.325615,69386
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40306,380000.0,69388,NaN,2,Appartement,61.0,3.0,NaN,4.886458,45.724246,2022,3,6229.508197,20.333333,4.127134,5.926675,69388
100089,405000.0,69385,NaN,1,Appartement,103.0,4.0,NaN,4.788276,45.751065,2025,11,3932.038835,25.750000,4.644391,3.959838,69385
100139,405000.0,69385,NaN,1,Appartement,103.0,4.0,NaN,4.788276,45.751065,2025,11,3932.038835,25.750000,4.644391,3.959838,69385
55120,410000.0,69091,NaN,0,Maison,139.0,5.0,793.0,4.763034,45.595210,2023,3,2949.640288,27.800000,4.941642,19.595788,69091


Nombre de doublons strictement identiques : 35


In [6]:
ligne1 = df_dvf.loc[35997]
ligne2 = df_dvf.loc[36000]

comparaison = pd.DataFrame({
    "ligne_35997": ligne1,
    "ligne_36000": ligne2,
    "identique": ligne1.eq(ligne2) | (ligne1.isna() & ligne2.isna())
})

display(comparaison)

,ligne_35997,ligne_36000,identique
valeur_fonciere,48500.0,48500.0,True
code_commune,69019,69019,True
lot1_surface_carrez,NaN,NaN,True
nombre_lots,1,1,True
type_local,Appartement,Appartement,True
surface_reelle_bati,25.0,25.0,True
nombre_pieces_principales,1.0,1.0,True
surface_terrain,NaN,NaN,True
longitude,4.74952,4.74952,True
latitude,46.1078,46.1078,True
